In [ ]:
######################################## Pattern Recognition Project ########################################
# Group 4

# Anomaly Localization using Diffusion Models
# Original paper: https://arxiv.org/pdf/2207.12268.pdf

# Vozinaki Anthi 03119168
# Giadikiaroglou Panagiotis 03119185
# Karafyllis Nikolas 03119890
# Litsos Ioannis 03119135
# Michelakis Panagiotis 03119119

#############################################################################################################

In [ ]:
import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
!python

In [ ]:
import torch
x = torch.rand(5, 3)
print(x)

In [ ]:
!wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb

In [ ]:
!sudo dpkg -i cuda-keyring_1.1-1_all.deb

In [ ]:
!sudo apt-get update

In [ ]:
!sudo apt install python3-pip

In [ ]:
!sudo apt-get -y install cuda-toolkit-12-3

In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
!pip install monai

In [ ]:
!nvidia-smi

In [ ]:
!python -m pip install -U pip
!python -m pip install -U matplotlib
!python -m pip install -U notebook

In [ ]:
!pip install lpips
!pip install monai-generative==0.2.2

In [ ]:
!git clone https://github.com/Project-MONAI/GenerativeModels.git

In [ ]:
!wget https://github.com/Project-MONAI/MONAI/blob/dev/setup.py

In [ ]:
!cd GenerativeModels/
!python setup.py install

In [ ]:
!BUILD_MONAI=1 pip install --no-build-isolation git+https://github.com/Project-MONAI/MONAI#egg=monai

In [ ]:
!python -c "import monai; monai.config.print_config()"

In [ ]:
!python -c "import monai" || pip install -q "monai-weekly[tqdm]"
!python -c "import matplotlib" || pip install -q matplotlib
%matplotlib inline

This experiment constitues a reproduction of the  Sanchez et al. What is Healthy? Generative Counterfactual Diffusion for Lesion Localization. DGM 4 MICCAI 2022.

In particular, by leveraging recent advances in generative diffusion probabilistic models, we synthesize counterfactuals of "How would a patient appear if X pathology was not present?". The difference image between the observed patient state and the healthy counterfactual can be used for inferring the location of pathology. We generate counterfactuals that correspond to the minimal change of the input such that it is transformed to healthy domain. We create these counterfactual diffusion models by manipulating the generation process with implicit guidance.

In summary, the tutorial will cover the following:

Loading and preprocessing a dataset (we extract the brain MRI dataset 2D slices from 3D volumes from the BraTS dataset)
Training a 2D diffusion model
Anomaly detection with the trained model

This method results in anomaly heatmaps. It is weakly supervised. The information about labels is not fed to the model as segmentation masks but as a scalar signal (is there an anomaly or not), which is used to guide the diffusion process.

During inference, the model generates a counterfactual image, which is then compared to the original image. The difference between the two images is used to generate an anomaly heatmap.



In [ ]:
import os
import tempfile
import time
import os
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import sys
from monai import transforms
from monai.apps import DecathlonDataset
from monai.config import print_config
from monai.data import DataLoader
from monai.utils import set_determinism
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm


from generative.inferers import DiffusionInferer
from generative.networks.nets.diffusion_model_unet import DiffusionModelUNet
from generative.networks.schedulers.ddim import DDIMScheduler

torch.multiprocessing.set_sharing_strategy("file_system")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print_config()

In [ ]:
directory = os.environ.get("MONAI_DATA_DIRECTORY")
root_dir = tempfile.mkdtemp() if directory is None else directory

In [ ]:
set_determinism(42)


In [ ]:
channel = 0  # 0 = Flair
assert channel in [0, 1, 2, 3], "Choose a valid channel"

train_transforms = transforms.Compose(
    [
        transforms.LoadImaged(keys=["image", "label"]),
        transforms.EnsureChannelFirstd(keys=["image", "label"]),
        transforms.Lambdad(keys=["image"], func=lambda x: x[channel, :, :, :]),
        transforms.EnsureChannelFirstD(channel_dim="no_channel",keys=["image"]),
        transforms.EnsureTyped(keys=["image", "label"]),
        transforms.Orientationd(keys=["image", "label"], axcodes="RAS"),
        transforms.Spacingd(keys=["image", "label"], pixdim=(3.0, 3.0, 2.0), mode=("bilinear", "nearest")),
        transforms.CenterSpatialCropd(keys=["image", "label"], roi_size=(64, 64, 44)),
        transforms.ScaleIntensityRangePercentilesd(keys="image", lower=0, upper=99.5, b_min=0, b_max=1),
        transforms.RandSpatialCropd(keys=["image", "label"], roi_size=(64, 64, 1), random_size=False),
        transforms.Lambdad(keys=["image", "label"], func=lambda x: x.squeeze(-1)),
        transforms.CopyItemsd(keys=["label"], times=1, names=["slice_label"]),
        transforms.Lambdad(keys=["slice_label"], func=lambda x: 2.0 if x.sum() > 0 else 1.0),
    ]
)

We download the BraTS dataset (if already downloaded set the "download" boolean variable to False.

In [ ]:
train_ds = DecathlonDataset(
    root_dir=root_dir,
    task="Task01_BrainTumour",
    section="training",
    cache_rate=1.0,  # you may need a few Gb of RAM... Set to 0 otherwise
    num_workers=4,
    download=True,  # Set download to True if the dataset hasnt been downloaded yet
    seed=0,
    transform=train_transforms,
)
print(f"Length of training data: {len(train_ds)}")
print(f'Train image shape {train_ds[0]["image"].shape}')

val_ds = DecathlonDataset(
    root_dir=root_dir,
    task="Task01_BrainTumour",
    section="validation",
    cache_rate=1.0,  # you may need a few Gb of RAM... Set to 0 otherwise
    num_workers=4,
    download=True,  # Set download to True if the dataset hasnt been downloaded yet
    seed=0,
    transform=train_transforms,
)
print(f"Length of training data: {len(val_ds)}")
print(f'Validation Image shape {val_ds[0]["image"].shape}')

We define the architecture of the UNet neural network that our diffusion model will use. (notice it has wo attention layers).

In [ ]:
device = torch.device("cuda")
embedding_dimension = 64
model = DiffusionModelUNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    num_channels=(64, 64, 64),
    attention_levels=(False, True, True),
    num_res_blocks=1,
    num_head_channels=16,
    with_conditioning=True,
    cross_attention_dim=embedding_dimension,
).to(device)
embed = torch.nn.Embedding(num_embeddings=3, embedding_dim=embedding_dimension, padding_idx=0).to(device)

scheduler = DDIMScheduler(num_train_timesteps=1000)
optimizer = torch.optim.Adam(params=list(model.parameters()) + list(embed.parameters()), lr=1e-5)

inferer = DiffusionInferer(scheduler)

We will use classifier-free guidance

In [ ]:
condition_dropout = 0.15
n_iterations = 1e4
batch_size = 32
val_interval = 100
iter_loss_list = []
val_iter_loss_list = []
iterations = []
iteration = 0
iter_loss = 0

train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True, persistent_workers=True
)
val_loader = DataLoader(
    val_ds, batch_size=batch_size, shuffle=False, num_workers=4, drop_last=True, persistent_workers=True
)

scaler = GradScaler()
total_start = time.time()

while iteration < n_iterations:
    for batch in train_loader:
        iteration += 1
        model.train()
        images, classes = batch["image"].to(device), batch["slice_label"].to(device)
        # 15% of the time, class conditioning dropout
        classes = classes * (torch.rand_like(classes) > condition_dropout)
        # cross attention expects shape [batch size, sequence length, channels]
        class_embedding = embed(classes.long().to(device)).unsqueeze(1)
        optimizer.zero_grad(set_to_none=True)
        # pick a random time step t
        timesteps = torch.randint(0, 1000, (len(images),)).to(device)

        with autocast(enabled=True):
            # Generate random noise
            noise = torch.randn_like(images).to(device)
            # Get model prediction
            noise_pred = inferer(
                inputs=images, diffusion_model=model, noise=noise, timesteps=timesteps, condition=class_embedding
            )
            loss = F.mse_loss(noise_pred.float(), noise.float())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        iter_loss += loss.item()
        sys.stdout.write(f"Iteration {iteration}/{n_iterations} - train Loss {loss.item():.4f}" + "\r")
        sys.stdout.flush()

        if (iteration) % val_interval == 0:
            model.eval()
            val_iter_loss = 0
            for val_step, val_batch in enumerate(val_loader):
                images, classes = val_batch["image"].to(device), val_batch["slice_label"].to(device)
                # cross attention expects shape [batch size, sequence length, channels]
                class_embedding = embed(classes.long().to(device)).unsqueeze(1)
                timesteps = torch.randint(0, 1000, (len(images),)).to(device)
                with torch.no_grad():
                    with autocast(enabled=True):
                        noise = torch.randn_like(images).to(device)
                        noise_pred = inferer(
                            inputs=images,
                            diffusion_model=model,
                            noise=noise,
                            timesteps=timesteps,
                            condition=class_embedding,
                        )
                        val_loss = F.mse_loss(noise_pred.float(), noise.float())
                val_iter_loss += val_loss.item()
            iter_loss_list.append(iter_loss / val_interval)
            val_iter_loss_list.append(val_iter_loss / (val_step + 1))
            iterations.append(iteration)
            iter_loss = 0
            print(
                f"Train Loss {loss.item():.4f}, Interval Loss {iter_loss_list[-1]:.4f}, Interval Loss Val {val_iter_loss_list[-1]:.4f}"
            )


total_time = time.time() - total_start

print(f"train diffusion completed, total time: {total_time}.")

plt.style.use("seaborn-bright")
plt.title("Learning Curves Diffusion Model", fontsize=20)
plt.plot(iterations, iter_loss_list, color="C0", linewidth=2.0, label="Train")
plt.plot(
    iterations, val_iter_loss_list, color="C1", linewidth=2.0, label="Validation"
)  # np.linspace(1, n_iterations, len(val_iter_loss_list))
plt.yticks(fontsize=12), plt.xticks(fontsize=12)
plt.xlabel("Iterations", fontsize=16), plt.ylabel("Loss", fontsize=16)
plt.legend(prop={"size": 14})
plt.show()

The cell below is only used for verification purposes in order to ensure that classifier-gree guidance works.

In [ ]:
model.eval()
scheduler.clip_sample = True
guidance_scale = 3
conditioning = torch.cat([torch.zeros(1).long(), 2 * torch.ones(1).long()], dim=0).to(
    device
)  # 2*torch.ones(1).long() is the class label for the UNHEALTHY (tumor) class
class_embedding = embed(conditioning).unsqueeze(
    1
)  # cross attention expects shape [batch size, sequence length, channels]
noise = torch.randn((1, 1, 64, 64))
noise = noise.to(device)
scheduler.set_timesteps(num_inference_steps=100)
progress_bar = tqdm(scheduler.timesteps)
for t in progress_bar:
    with autocast(enabled=True):
        with torch.no_grad():
            noise_input = torch.cat([noise] * 2)
            model_output = model(noise_input, timesteps=torch.Tensor((t,)).to(noise.device), context=class_embedding)
            noise_pred_uncond, noise_pred_text = model_output.chunk(2)
            noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

    noise, _ = scheduler.step(noise_pred, t, noise)

plt.style.use("default")
plt.imshow(noise[0, 0].cpu(), vmin=0, vmax=1, cmap="gray")
plt.tight_layout()
plt.axis("off")
plt.show()

We pick a diseased subject of the validation set as input image. We want to translate it to its healthy reconstruction.



In [ ]:
#idx_unhealthy = np.argwhere(val_batch["slice_label"].numpy() == 2).squeeze()

#idx = idx_unhealthy[15]  # Change this index in order to choose a different image (the index goes up to 20)
#inputting = val_batch["image"][idx]  # Pick an input slice of the validation set to be transformed
#inputlabel = val_batch["slice_label"][idx]  # Check whether it is healthy or diseased

############### Use the following lines for calculating the DICE score. Use the above lines for proper evaluation #######################
data_sample = val_ds[22]  # Get the first sample from the dataset

image = data_sample["image"]  # This is your MRI slice
inputting = data_sample["image"]
inputlabel = data_sample["label"]  # This is the corresponding mask

###########################################################################################################3333


plt.figure("input" + str(inputlabel))
plt.imshow(inputting[0], vmin=0, vmax=1, cmap="gray")
plt.axis("off")
plt.tight_layout()
plt.show()

model.eval()
print("input label: ", inputlabel)

The image-to-image translation has two steps
Encoding the input image into a latent space with the reversed DDIM sampling scheme
Sampling from the latent space using gradient guidance towards the desired class label y=1 (healthy)
In order to sample using gradient guidance, we first need to encode the input image in noise by using the reversed DDIM sampling scheme. We define the number of steps in the noising and denoising process by L.

In [ ]:
model.eval()

guidance_scale = 3.0
total_timesteps = 500
latent_space_depth = int(total_timesteps * 0.25)

current_img = inputting[None, ...].to(device)
scheduler.set_timesteps(num_inference_steps=total_timesteps)

## Encoding

scheduler.clip_sample = False
class_embedding = embed(torch.zeros(1).long().to(device)).unsqueeze(1)
progress_bar = tqdm(range(latent_space_depth))
for i in progress_bar:  # go through the noising process
    t = i
    with torch.no_grad():
        model_output = model(current_img, timesteps=torch.Tensor((t,)).to(current_img.device), context=class_embedding)
    current_img, _ = scheduler.reversed_step(model_output, t, current_img)
    progress_bar.set_postfix({"timestep input": t})

latent_img = current_img

## Decoding
conditioning = torch.cat([torch.zeros(1).long(), torch.ones(1).long()], dim=0).to(device)
class_embedding = embed(conditioning).unsqueeze(1)

progress_bar = tqdm(range(latent_space_depth))
for i in progress_bar:  # go through the denoising process
    t = latent_space_depth - i
    current_img_double = torch.cat([current_img] * 2)
    with torch.no_grad():
        model_output = model(
            current_img_double, timesteps=torch.Tensor([t, t]).to(current_img.device), context=class_embedding
        )
    noise_pred_uncond, noise_pred_text = model_output.chunk(2)
    noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)
    current_img, _ = scheduler.step(noise_pred, t, current_img)
    progress_bar.set_postfix({"timestep input": t})
    torch.cuda.empty_cache()

Visualize the result of our process

In [ ]:
def visualize(img):
    _min = img.min()
    _max = img.max()
    normalized_img = (img - _min) / (_max - _min)
    return normalized_img


diff = abs(inputting.cpu() - current_img[0].cpu()).detach().numpy()
row = 4
plt.style.use("default")

fig = plt.figure(figsize=(10, 30))

ax = plt.subplot(1, row, 2)
ax.imshow(latent_img[0, 0].cpu().detach().numpy(), vmin=0, vmax=1, cmap="gray")
ax.set_title("Latent Image"), plt.tight_layout(), plt.axis("off")

ax = plt.subplot(1, row, 3)
ax.imshow(current_img[0, 0].cpu().detach().numpy(), vmin=0, vmax=1, cmap="gray")
ax.set_title("Reconstructed \n Image"), plt.tight_layout(), plt.axis("off")


ax = plt.subplot(1, row, 4)
ax.imshow(diff[0], cmap="inferno")
ax.set_title("Anomaly Map"), plt.tight_layout(), plt.axis("off")

ax = plt.subplot(1, row, 1)
ax.imshow(inputting[0], vmin=0, vmax=1, cmap="gray")
ax.set_title("Original Image"), plt.tight_layout(), plt.axis("off")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

print(len(val_batch))
#data_sample = val_ds[7]  # Get the first sample from the dataset

image = inputting  # This is your MRI slice
mask = data_sample["label"]  # This is the corresponding mask


if hasattr(image, "cpu"):
    image = image.cpu().numpy()
if hasattr(mask, "cpu"):
    mask = mask.cpu().numpy()

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(image[0, :, :], cmap='gray')
plt.title('MRI Slice')

plt.subplot(1, 2, 2)
plt.imshow(mask[0, :, :], cmap='gray')
plt.title('Segmentation Mask')

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import filters

def generate_masks_from_difference(diff, percentile=98, show_plots=False):
    # Manual thresholding
    threshold_value = np.percentile(diff, percentile)  # Adjust the percentile as needed
    manual_threshold_mask = (diff > threshold_value).astype(np.uint8)

    # Otsu's thresholding
    otsu_threshold = filters.threshold_otsu(diff)
    otsu_mask = (diff > otsu_threshold).astype(np.uint8)

    if show_plots:
        fig, ax = plt.subplots(1, 3, figsize=(15, 5))
        ax[0].imshow(diff, cmap='gray')
        ax[0].set_title('Difference Image'), plt.axis('off')

        ax[1].imshow(manual_threshold_mask, cmap='gray')
        ax[1].set_title('Binary Mask (Manual Threshold)'), plt.axis('off')

        ax[2].imshow(otsu_mask, cmap='gray')
        ax[2].set_title('Otsu Threshold Mask'), plt.axis('off')

        plt.tight_layout()
        plt.show()

    return manual_threshold_mask, otsu_mask

manual_mask, otsu_mask = generate_masks_from_difference(diff[0], show_plots=True)



In [ ]:
def dice_score(true_mask, pred_mask):
    true_mask_bool = true_mask.astype(bool)
    pred_mask_bool = pred_mask.astype(bool)

    # Intersection and union calculations
    intersection = np.logical_and(true_mask_bool, pred_mask_bool).sum()
    true_sum = true_mask_bool.sum()
    pred_sum = pred_mask_bool.sum()

    # Dice score calculation
    if true_sum + pred_sum == 0:  # To avoid division by zero if both masks are empty
        return 1.0
    dice = (2. * intersection) / (true_sum + pred_sum)

    return dice

dice_score_manual = dice_score(mask, manual_mask)
dice_score_otsu = dice_score(mask, otsu_mask)

print(f"Dice Score (Manual Threshold): {dice_score_manual}")
print(f"Dice Score (Otsu Threshold): {dice_score_otsu}")